In [10]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from scipy import stats
from creditcard_psp.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

from creditcard_psp.dataset import load_transactions, merge_service_fees, assign_transaction_ids
from creditcard_psp.features import add_time_features

input_path  = RAW_DATA_DIR / "PSP_Jan_Feb_2019.xlsx"
fee_path    = RAW_DATA_DIR / "PSP_Servicegebuehren.xlsx"

In [11]:
# Load data
df = load_transactions(input_path)

# Count rows and delete duplicates
print(df.count())

duplicates = df.duplicated().sum()
print("Anzahl Duplikate: ", duplicates)

df = df.drop_duplicates()
print(df.count())

2025-10-04 16:46:07.766 | INFO     | creditcard_psp.dataset:load_transactions:28 - Loading data from C:\Users\miria\creditcard_psp\data\raw\PSP_Jan_Feb_2019.xlsx
tmsp          50410
country       50410
amount        50410
success       50410
PSP           50410
3D_secured    50410
card          50410
dtype: int64
Anzahl Duplikate:  81
tmsp          50329
country       50329
amount        50329
success       50329
PSP           50329
3D_secured    50329
card          50329
dtype: int64


In [12]:
# Merge Service-Fees
df = merge_service_fees(df, fee_path)

2025-10-04 16:46:29.417 | INFO     | creditcard_psp.dataset:merge_service_fees:112 - Merging service fees from C:\Users\miria\creditcard_psp\data\raw\PSP_Servicegebuehren.xlsx
2025-10-04 16:46:29.527 | WARNING  | creditcard_psp.dataset:merge_service_fees:128 - Missing fee entries after merge: {'fee_successful': 0, 'fee_not_successful': 0}


In [13]:
# Extract weekday, hour, minute
df = add_time_features(df)

In [14]:
# Adds transaction_id, transaction_success, attempt_number
df = assign_transaction_ids(df)      

In [15]:
tmp = df.copy()
tmp["minute_bucket"] = pd.to_datetime(tmp["tmsp"]).dt.floor("T")
ok = (tmp.groupby(["country","amount","minute_bucket"])["transaction_id"].nunique() <= 1).all()
print("Rule satisfied (same minute+country+amount -> single transaction_id):", ok)

Rule satisfied (same minute+country+amount -> single transaction_id): True


In [16]:
# attempt_number sollte innerhalb einer Transaktion bei 1 starten und hochzählen
mn_attempt = df.groupby("transaction_id")["attempt_number"].min().eq(1).all()
mono_attempt = df.sort_values(["transaction_id","tmsp"]) \
                 .groupby("transaction_id")["attempt_number"] \
                 .apply(lambda s: (s.diff().fillna(1) >= 0).all()).all()

# transaction_success = max(success) je Transaktion (falls 'success' existiert)
ok_succ = True
if "success" in df.columns:
    ok_succ = df.groupby("transaction_id")["success"].max().equals(
        df.groupby("transaction_id")["transaction_success"].max()
    )

print(f"attempt_number starts at 1: {mn_attempt}")
print(f"attempt_number monotonic:   {mono_attempt}")
print(f"transaction_success ok:     {ok_succ}")

attempt_number starts at 1: True
attempt_number monotonic:   True
transaction_success ok:     True


In [19]:
# Save DataFrame
df.to_pickle(PROCESSED_DATA_DIR / 'df.pkl')